# TASK#1

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

VIDEO_PATH = "/content/drive/MyDrive/check.mp4"
print("Exists:", os.path.exists(VIDEO_PATH))

Exists: True


In [8]:
# check if T4 GPU is working
import torch
print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [9]:

import os
import time
import cv2
import psutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO

# -----------------------------
# User Config
# -----------------------------
VIDEO_PATH = "/content/drive/MyDrive/check.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/YOLO_Outputs"
IMG_SIZE = 640
CONF_THRESH = 0.25
IOU_THRESH = 0.45
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Optional: if you have a labeled dataset YAML, set this to compute mAP/Precision/Recall.
# Example: DATA_YAML = "/content/coco128.yaml"
DATA_YAML = "coco128.yaml"

# Change yolo12 weight if your file name differs (e.g., custom yolo12 best.pt).
MODEL_WEIGHTS = {
    "YOLOv5": "yolov5su.pt",
    "YOLOv8": "yolov8n.pt",
    "YOLOv12": "yolo12n.pt",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
try:
    import pynvml
    pynvml.nvmlInit()
    _GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
except Exception:
    _GPU_HANDLE = None


def get_gpu_util_percent():
    if _GPU_HANDLE is None:
        return np.nan
    try:
        util = pynvml.nvmlDeviceGetUtilizationRates(_GPU_HANDLE)
        return float(util.gpu)
    except Exception:
        return np.nan


def iou_xyxy(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0.0, (a[2] - a[0])) * max(0.0, (a[3] - a[1]))
    area_b = max(0.0, (b[2] - b[0])) * max(0.0, (b[3] - b[1]))
    union = area_a + area_b - inter

    if union <= 0:
        return 0.0
    return inter / union


def greedy_match_ious(prev_boxes, curr_boxes, iou_thr=0.3):
    if len(prev_boxes) == 0 or len(curr_boxes) == 0:
        return []

    ious = []
    used_prev = set()
    used_curr = set()

    while True:
        best_iou = -1.0
        best_pair = None

        for i, pb in enumerate(prev_boxes):
            if i in used_prev:
                continue
            for j, cb in enumerate(curr_boxes):
                if j in used_curr:
                    continue
                iou = iou_xyxy(pb, cb)
                if iou > best_iou:
                    best_iou = iou
                    best_pair = (i, j)

        if best_pair is None or best_iou < iou_thr:
            break

        i, j = best_pair
        used_prev.add(i)
        used_curr.add(j)
        ious.append(best_iou)

    return ious


def draw_detections(frame, boxes, confs, clss, names):
    for box, conf, cls_id in zip(boxes, confs, clss):
        x1, y1, x2, y2 = [int(v) for v in box]
        cls_name = names.get(int(cls_id), str(int(cls_id)))
        label = f"{cls_name} {conf:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (60, 200, 60), 2)
        cv2.putText(
            frame,
            label,
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 0),
            2,
            cv2.LINE_AA,
        )
        cv2.putText(
            frame,
            label,
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )
    return frame


def get_model_size_mb(model, weight_hint):
    candidates = []
    if weight_hint:
        candidates.append(Path(weight_hint))
    for attr in ["ckpt_path", "pt_path", "model_path"]:
        v = getattr(model, attr, None)
        if isinstance(v, str):
            candidates.append(Path(v))

    if hasattr(model, "model") and hasattr(model.model, "pt_path"):
        v = getattr(model.model, "pt_path", None)
        if isinstance(v, str):
            candidates.append(Path(v))

    for p in candidates:
        try:
            if p.exists() and p.is_file():
                return round(p.stat().st_size / (1024 * 1024), 2)
        except Exception:
            pass
    return np.nan


def maybe_compute_quality_metrics(model, data_yaml, device):
    out = {
        "mAP@0.5": np.nan,
        "mAP@0.95": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
    }
    if not data_yaml:
        return out
    # Allow names like 'coco128.yaml' so Ultralytics can resolve/download dataset automatically.

    try:
        val = model.val(data=data_yaml, imgsz=IMG_SIZE, device=device, verbose=False)
        out["mAP@0.5"] = float(val.box.map50)
        out["mAP@0.95"] = float(val.box.map)
        out["Precision"] = float(val.box.mp)
        out["Recall"] = float(val.box.mr)
    except Exception as e:
        print(f"[WARN] Could not compute quality metrics: {e}")
    return out


def run_benchmark(model_name, model, weight_name, video_path, output_dir, device):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_src = cap.get(cv2.CAP_PROP_FPS)
    if fps_src <= 0:
        fps_src = 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_video = os.path.join(output_dir, f"annotated_{model_name}.mp4")
    writer = cv2.VideoWriter(
        out_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps_src,
        (width, height),
    )

    latencies_ms = []
    cpu_utils = []
    gpu_utils = []
    matched_ious = []

    prev_by_class = defaultdict(list)

    t_start = time.perf_counter()
    pbar = tqdm(total=total_frames if total_frames > 0 else None, desc=f"{model_name}")

    processed = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        t0 = time.perf_counter()
        result = model.predict(
            source=frame,
            imgsz=IMG_SIZE,
            conf=CONF_THRESH,
            iou=IOU_THRESH,
            device=device,
            verbose=False,
        )[0]
        t1 = time.perf_counter()

        latency = (t1 - t0) * 1000.0
        latencies_ms.append(latency)
        cpu_utils.append(psutil.cpu_percent(interval=None))
        gpu_utils.append(get_gpu_util_percent())

        boxes = result.boxes.xyxy.detach().cpu().numpy() if result.boxes is not None else np.empty((0, 4))
        confs = result.boxes.conf.detach().cpu().numpy() if result.boxes is not None else np.empty((0,))
        clss = result.boxes.cls.detach().cpu().numpy().astype(int) if result.boxes is not None else np.empty((0,), dtype=int)
        names = result.names

        # Temporal consistency from class-wise IoU matching with previous frame.
        curr_by_class = defaultdict(list)
        for box, cls_id in zip(boxes, clss):
            curr_by_class[int(cls_id)].append(box.tolist())

        all_cls = set(prev_by_class.keys()) | set(curr_by_class.keys())
        for cls_id in all_cls:
            ious = greedy_match_ious(prev_by_class.get(cls_id, []), curr_by_class.get(cls_id, []), iou_thr=0.3)
            matched_ious.extend(ious)
        prev_by_class = curr_by_class

        frame_annotated = draw_detections(frame.copy(), boxes, confs, clss, names)
        writer.write(frame_annotated)

        processed += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    total_time = time.perf_counter() - t_start
    avg_fps = processed / total_time if total_time > 0 else np.nan
    avg_latency = float(np.mean(latencies_ms)) if latencies_ms else np.nan
    avg_cpu = float(np.nanmean(cpu_utils)) if cpu_utils else np.nan
    avg_gpu = float(np.nanmean(gpu_utils)) if gpu_utils else np.nan

    temporal_consistency = float(np.mean(matched_ious)) if matched_ious else np.nan
    # Higher is better; low variation in frame-to-frame IoU implies more stable boxes.
    bbox_stability = float(np.clip(1.0 - np.std(matched_ious), 0.0, 1.0)) if matched_ious else np.nan

    model_size_mb = get_model_size_mb(model, weight_name)

    return {
        "Model": model_name,
        "FPS": round(avg_fps, 2),
        "Latency (ms)": round(avg_latency, 2),
        "GPU Utilization (%)": round(avg_gpu, 2) if not np.isnan(avg_gpu) else np.nan,
        "CPU Utilization (%)": round(avg_cpu, 2) if not np.isnan(avg_cpu) else np.nan,
        "Model Size (MB)": model_size_mb,
        "Bounding-box Stability": round(bbox_stability, 4) if not np.isnan(bbox_stability) else np.nan,
        "Temporal Consistency": round(temporal_consistency, 4) if not np.isnan(temporal_consistency) else np.nan,
        "Annotated Video": out_video,
    }


# -----------------------------
# Main Run
# -----------------------------
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

rows = []
for model_name, weight_name in MODEL_WEIGHTS.items():
    print(f"\n=== Loading {model_name} ({weight_name}) ===")
    try:
        model = YOLO(weight_name)
    except Exception as e:
        print(f"[ERROR] Failed to load {model_name} with '{weight_name}': {e}")
        print("Set MODEL_WEIGHTS to correct checkpoint path/name and rerun.")
        continue

    run_row = run_benchmark(
        model_name=model_name,
        model=model,
        weight_name=weight_name,
        video_path=VIDEO_PATH,
        output_dir=OUTPUT_DIR,
        device=DEVICE,
    )

    quality = maybe_compute_quality_metrics(model, DATA_YAML, DEVICE)
    run_row.update(quality)

    # Instructor-friendly remarks template
    if model_name == "YOLOv5":
        run_row["Remarks"] = "Balanced between speed and accuracy"
    elif model_name == "YOLOv8":
        run_row["Remarks"] = "Generally faster with strong real-time performance"
    else:
        run_row["Remarks"] = "Check if YOLOv12 weights are available and compatible"

    rows.append(run_row)

if not rows:
    raise RuntimeError("No model ran successfully. Check MODEL_WEIGHTS and environment setup.")

summary_df = pd.DataFrame(rows)[
    [
        "Model",
        "FPS",
        "Latency (ms)",
        "mAP@0.5",
        "mAP@0.95",
        "Precision",
        "Recall",
        "GPU Utilization (%)",
        "CPU Utilization (%)",
        "Model Size (MB)",
        "Bounding-box Stability",
        "Temporal Consistency",
        "Remarks",
        "Annotated Video",
    ]
]

csv_path = os.path.join(OUTPUT_DIR, "performance_summary.csv")
summary_df.to_csv(csv_path, index=False)

print("\nBenchmark complete.")
print(f"Summary CSV: {csv_path}")
display(summary_df)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

=== Loading YOLOv5 (yolov5su.pt) ===


YOLOv5:   0%|          | 0/451 [00:00<?, ?it/s]

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

WARNING ⚠️ Dataset 'coco128.yaml' images not found, missing path '/content/datasets/coco128/images/train2017'
Unzipping /content/datasets/coco128.zip to /content/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 4.6Kfiles/s 0.1s
Dataset download success ✅ (0.5s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1113.2±250.6 MB/s, size: 42.7 KB)
val: Scanning /content/datasets/coco128/labels/train2017... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 1.5Kit/s 0.1s
val: New cache created: /content/datasets/coco128/labels/train2017.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.2ss
                   all        128        929      0.796      0.629      0.737      0.563
Speed: 2.3ms preprocess, 8.7ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/runs/

YOLOv8:   0%|          | 0/451 [00:00<?, ?it/s]

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1158.4±294.5 MB/s, size: 50.2 KB)
val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 35.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.6it/s 3.1s0.2ss
                   all        128        929      0.639      0.536      0.607      0.448
Speed: 3.7ms preprocess, 6.6ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to /content/runs/detect/val2

=== Loading YOLOv12 (yolo12n.pt) ===


YOLOv12:   0%|          | 0/451 [00:00<?, ?it/s]

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1406.6±508.2 MB/s, size: 59.2 KB)
val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 59.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.0it/s 4.1s0.3ss
                   all        128        929       0.71      0.622      0.687      0.529
Speed: 5.1ms preprocess, 9.9ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/detect/val3

Benchmark complete.
Summary CSV: /content/drive/MyDrive/YOLO_Outputs/performance_summary.csv


,Model,FPS,Latency (ms),mAP@0.5,mAP@0.95,Precision,Recall,GPU Utilization (%),CPU Utilization (%),Model Size (MB),Bounding-box Stability,Temporal Consistency,Remarks,Annotated Video
0,YOLOv5,6.00,23.35,0.737091,0.563056,0.795922,0.629298,6.14,81.85,17.72,0.8569,0.8135,Balanced between speed and accuracy,/content/drive/MyDrive/YOLO_Outputs/annotated_...
1,YOLOv8,6.31,18.02,0.607167,0.447785,0.638502,0.536086,3.75,82.16,6.25,0.8644,0.8223,Generally faster with strong real-time perform...,/content/drive/MyDrive/YOLO_Outputs/annotated_...
2,YOLOv12,6.11,29.39,0.686608,0.528709,0.709673,0.621850,5.03,81.33,5.34,0.8644,0.8253,Check if YOLOv12 weights are available and com...,/content/drive/MyDrive/YOLO_Outputs/annotated_...


# TASK#2

In [10]:
# Task 2 runner: sequential hyperparameter experiments
import time
import os
import numpy as np
import pandas as pd
import psutil
from ultralytics import YOLO

# Keep these paths as needed
VIDEO_PATH = "/content/drive/MyDrive/check.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/YOLO_Outputs"
DATA_YAML = "coco128.yaml"  # set to your dataset yaml for train/val metrics
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set >0 to train in each experiment, 0 for inference-only quick comparison
TRAIN_EPOCHS = 5

# Model weights (edit if your YOLOv12 checkpoint name differs)
MODEL_PATHS = {
    "YOLOv5": "yolov5su.pt",
    "YOLOv8": "yolov8n.pt",
    "YOLOv12": "yolo12n.pt",
}

# Required sequential experiments from your lab sheet
EXPERIMENTS = [
    {"Exp ID": "Exp-1", "Model": "YOLOv5",  "LR": 0.0010, "Batch": 16, "Image Size": 640, "Conf Thresh": 0.25, "IoU Thresh": 0.50},
    {"Exp ID": "Exp-2", "Model": "YOLOv8",  "LR": 0.0005, "Batch": 16, "Image Size": 640, "Conf Thresh": 0.25, "IoU Thresh": 0.50},
    {"Exp ID": "Exp-3", "Model": "YOLOv12", "LR": 0.0005, "Batch": 32, "Image Size": 512, "Conf Thresh": 0.25, "IoU Thresh": 0.50},
    {"Exp ID": "Exp-4", "Model": "YOLOv8",  "LR": 0.0008, "Batch": 16, "Image Size": 640, "Conf Thresh": 0.30, "IoU Thresh": 0.55},
]


def _gpu_util_percent():
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        return float(pynvml.nvmlDeviceGetUtilizationRates(h).gpu)
    except Exception:
        return np.nan


def benchmark_video_once(model, video_path, imgsz, conf, iou, device):
    import cv2

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Video not found or cannot open: {video_path}")

    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    lat_ms = []
    cpu_vals = []
    gpu_vals = []
    t_all_0 = time.perf_counter()
    processed = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        t0 = time.perf_counter()
        _ = model.predict(
            source=frame,
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            device=device,
            verbose=False,
        )[0]
        t1 = time.perf_counter()

        lat_ms.append((t1 - t0) * 1000.0)
        cpu_vals.append(psutil.cpu_percent(interval=None))
        gpu_vals.append(_gpu_util_percent())
        processed += 1

    cap.release()

    t_all = time.perf_counter() - t_all_0
    fps = (processed / t_all) if t_all > 0 else np.nan
    avg_lat = float(np.mean(lat_ms)) if lat_ms else np.nan
    avg_gpu = float(np.nanmean(gpu_vals)) if gpu_vals else np.nan

    return {
        "frames": processed,
        "video_frames": n,
        "fps": fps,
        "latency_ms": avg_lat,
        "gpu_pct": avg_gpu,
    }


def maybe_train_and_validate(model, exp_cfg, data_yaml, device):
    metrics = {
        "mAP@0.5": np.nan,
        "mAP@0.95": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
    }

    if not data_yaml:
        return metrics
    if TRAIN_EPOCHS <= 0:
        # Validation-only on current weights
        try:
            v = model.val(data=data_yaml, imgsz=exp_cfg["Image Size"], batch=exp_cfg["Batch"], device=device, verbose=False)
            metrics["mAP@0.5"] = float(v.box.map50)
            metrics["mAP@0.95"] = float(v.box.map)
            metrics["Precision"] = float(v.box.mp)
            metrics["Recall"] = float(v.box.mr)
        except Exception as e:
            print(f"[WARN] Validation skipped: {e}")
        return metrics

    # Train then validate
    try:
        _ = model.train(
            data=data_yaml,
            epochs=TRAIN_EPOCHS,
            imgsz=exp_cfg["Image Size"],
            batch=exp_cfg["Batch"],
            lr0=exp_cfg["LR"],
            device=device,
            verbose=False,
        )
        v = model.val(data=data_yaml, imgsz=exp_cfg["Image Size"], batch=exp_cfg["Batch"], device=device, verbose=False)
        metrics["mAP@0.5"] = float(v.box.map50)
        metrics["mAP@0.95"] = float(v.box.map)
        metrics["Precision"] = float(v.box.mp)
        metrics["Recall"] = float(v.box.mr)
    except Exception as e:
        print(f"[WARN] Train/Val skipped: {e}")

    return metrics


# Run sequentially
results = []
for exp in EXPERIMENTS:
    model_name = exp["Model"]
    model_path = MODEL_PATHS[model_name]
    print(f"\n=== {exp['Exp ID']} | {model_name} ===")

    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f"[ERROR] Could not load {model_name} ({model_path}): {e}")
        continue

    # Optional train/val metrics
    quality = maybe_train_and_validate(model, exp, DATA_YAML, "cuda:0" if __import__('torch').cuda.is_available() else "cpu")

    # Inference benchmark
    try:
        inf = benchmark_video_once(
            model=model,
            video_path=VIDEO_PATH,
            imgsz=exp["Image Size"],
            conf=exp["Conf Thresh"],
            iou=exp["IoU Thresh"],
            device="cuda:0" if __import__('torch').cuda.is_available() else "cpu",
        )
    except Exception as e:
        print(f"[ERROR] Inference failed in {exp['Exp ID']}: {e}")
        inf = {"fps": np.nan, "latency_ms": np.nan, "gpu_pct": np.nan}

    row = {
        "Exp ID": exp["Exp ID"],
        "Model": model_name,
        "LR": exp["LR"],
        "Batch": exp["Batch"],
        "Image Size": exp["Image Size"],
        "Conf Thresh": exp["Conf Thresh"],
        "IoU Thresh": exp["IoU Thresh"],
        "mAP@0.5": quality["mAP@0.5"],
        "mAP@0.95": quality["mAP@0.95"],
        "Precision": quality["Precision"],
        "Recall": quality["Recall"],
        "FPS": round(inf["fps"], 2) if not np.isnan(inf["fps"]) else np.nan,
        "GPU (%)": round(inf["gpu_pct"], 2) if not np.isnan(inf["gpu_pct"]) else np.nan,
        "Remarks": "Sequential hyperparameter run",
    }
    results.append(row)

if not results:
    raise RuntimeError("No experiment completed. Check video path and model checkpoints.")

task2_df = pd.DataFrame(results)[
    [
        "Exp ID", "Model", "LR", "Batch", "Image Size", "Conf Thresh", "IoU Thresh",
        "mAP@0.5", "mAP@0.95", "Precision", "Recall", "FPS", "GPU (%)", "Remarks",
    ]
]

task2_csv = os.path.join(OUTPUT_DIR, "hyperparameter_performance_summary.csv")
task2_df.to_csv(task2_csv, index=False)

print("Saved:", task2_csv)
display(task2_df)


=== Exp-1 | YOLOv5 ===
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5su.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

,Exp ID,Model,LR,Batch,Image Size,Conf Thresh,IoU Thresh,mAP@0.5,mAP@0.95,Precision,Recall,FPS,GPU (%),Remarks
0,Exp-1,YOLOv5,0.0010,16,640,0.25,0.50,0.770248,0.589684,0.812051,0.686115,16.27,17.13,Sequential hyperparameter run
1,Exp-2,YOLOv8,0.0005,16,640,0.25,0.50,0.659468,0.488920,0.680147,0.570087,16.42,10.27,Sequential hyperparameter run
2,Exp-3,YOLOv12,0.0005,32,512,0.25,0.50,0.546844,0.411267,0.600542,0.460562,15.40,11.14,Sequential hyperparameter run
3,Exp-4,YOLOv8,0.0008,16,640,0.30,0.55,0.659468,0.488920,0.680147,0.570087,16.38,10.20,Sequential hyperparameter run
